# Procurement Intelligence Assistant — MVP

RAG-based chatbot answering procurement, sourcing, and supply chain questions
from a curated set of YouTube video transcripts.

*Pipeline stages:* transcription (done) → chunking → embeddings → vector store → retrieval + LLM

## 1. Setup

In [1]:
# Core dependencies for chunking and token counting
import json
import tiktoken

# cl100k_base is the tokenizer used by GPT-3.5/4 family models —
# a reasonable proxy for token count even if using a different embedding model
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) ->int:
    return len(enc.encode(text))

## 2. Load Transcripts


#Load transcripts

Sourcedata: data/transcripts.json
Structure: {video_id: {title, segments: [{start, end, text}, ...]}}
#Two videos (uSjrTpJHn2g, U1E6rtnreoA) were pre-trimmed to remove webinar preamble/roll-call content before this file was saved.

In [4]:
with open("data/transcripts.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} videos")

Loaded 10 videos


## 3. Chunking

*Strategy:* merge consecutive Whisper segments into ~250-token chunks,
with ~40-token overlap so ideas that span a chunk boundary (e.g. a numbered
list) still appear complete in at least one chunk.

Each chunk keeps video_id, title, start, end as metadata —
needed later for citations and clickable timestamped YouTube links.

In [5]:
def count_tokens(text: str) -> int:
    """Return the token count of a string using the cl100k_base tokenizer."""
    return len(enc.encode(text))


def chunk_segments(segments, target_tokens=250, overlap_tokens=40):
    """
    Merge a list of {start, end, text} segments into token-bounded chunks.

    Args:
        segments: list of transcript segments for one video
        target_tokens: approx. token count to accumulate before closing a chunk
        overlap_tokens: approx. token count carried over into the next chunk

    Returns:
        list of {text, start, end, token_count} dicts
    """
    chunks = []
    current = []
    current_tokens = 0

    for seg in segments:
        current.append(seg)
        current_tokens += count_tokens(seg["text"])

        if current_tokens >= target_tokens:
            chunks.append(_build_chunk(current, current_tokens))
            current, current_tokens = _take_overlap(current, overlap_tokens)

    if current:
        chunks.append(_build_chunk(current, current_tokens))

    return chunks


def _build_chunk(seg_list, token_count):
    """Join a list of segments into a single chunk record."""
    return {
        "text": " ".join(s["text"].strip() for s in seg_list),
        "start": seg_list[0]["start"],
        "end": seg_list[-1]["end"],
        "token_count": token_count,
    }


def _take_overlap(seg_list, overlap_tokens):
    """Return the trailing segments (and their token count) to seed the next chunk."""
    overlap_seg_list = []
    overlap_count = 0
    for s in reversed(seg_list):
        overlap_count += count_tokens(s["text"])
        overlap_seg_list.insert(0, s)
        if overlap_count >= overlap_tokens:
            break
    return overlap_seg_list, overlap_count

## 4.Test chunk size in one video

In [6]:
test_video = "ZCIJQJRw6xw"

for target in [200, 250, 300]:
    test_chunks = chunk_segments(
        data[test_video]["segments"],
        target_tokens=target,
        overlap_tokens=int(target * 0.15)
    )
    print(f"\n=== target={target} tokens ({len(test_chunks)} chunks) ===")
    print(test_chunks[1]["text"])


=== target=200 tokens (13 chunks) ===
Whether it's software for a startup or machines for a factory, procurement decides who delivers what, when, and how much it costs. For example, think of procurement like planning a big wedding. You don't just go buy food and flowers. You plan your guest list, find reliable caterers, compare quotations, ensure everything arrives on time, track the budget, sign contracts. That's procurement on a business scale. Let's take a real-world example. Tata Motors, one of India's largest automobile manufacturers. To build each vehicle, they need tires from Bridgestone, steel from Tata Steel, electronics from Bosch, paint, seats, dashboards, software systems, etc. Tata Motors doesn't manufacture all of this in-house. Instead, they procure parts from a global network of suppliers who specialize in those products. If even one part, like a microchip, doesn't arrive on time, production halts, deadlines are missed, and millions are lost. That's how powerful procur

## 5. Lock in final size and run on all videos

In [8]:
FINAL_TARGET = 250   # <- update based on Step 4
FINAL_OVERLAP = 40    # <- update based on Step 4

all_chunks = []

for vid, content in data.items():
    video_chunks = chunk_segments(
        content["segments"],
        target_tokens=FINAL_TARGET,
        overlap_tokens=FINAL_OVERLAP
    )
    for c in video_chunks:
        c["video_id"] = vid
        c["title"] = content["title"]
        all_chunks.append(c)

print(f"Total chunks: {len(all_chunks)}")
print(f"Videos processed: {len(data)}")
print(f"Avg chunks per video: {len(all_chunks) / len(data):.1f}")

Total chunks: 160
Videos processed: 10
Avg chunks per video: 16.0


## 6. Sanity check the full set

In [9]:
import random

sample_chunks = random.sample(all_chunks, 5)

for c in sample_chunks:
    print(f"--- {c['title']} ({c['start']:.0f}s-{c['end']:.0f}s, {c['token_count']} tokens) ---")
    print(c["text"])
    print()

--- Contract Management in Procurement | Stages & Tools (129s-213s, 263 tokens) ---
Stage 4. Contract renewal or closure. Evaluate supplier performance and determine whether to extend or terminate the contract. Archive closed contracts and collect postmorm lessons. Example, a software company uses a 12-month SaaS subscription for a CRM tool. Near expiration, procurement evaluates usage, cost effectiveness, and vendor support to decide on renewal. Four, components of a procurement contract. Scope of work, SOW clear definition of the goods or services to be delivered. Pricing and payment terms include structure, fixed, milestone, hourly, due dates, and penalties. Service level agreements, SLAs, and KPIs, define expectations for delivery times, quality, and issue resolution. Termination clauses, outline under what circumstances the contract may be terminated early. Force majeure, provision to manage unforeseen disruptions, e.g. pandemics, natural disasters. Confidentiality and IP rights p

## 7.Save the chunks

In [10]:
with open("data/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print("Saved", len(all_chunks), "chunks to data/chunks.json")

Saved 160 chunks to data/chunks.json


## 8. Install dependencies for Embeddings

In [11]:
!pip install chromadb openai tiktoken


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 9. Setup

In [13]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from dotenv import load_dotenv
load_dotenv()

import json
import chromadb
from openai import OpenAI

client = OpenAI()  # now picks up OPENAI_API_KEY from .env
chroma_client = chromadb.PersistentClient(path="./chroma_db")

## 10.Loading chunks

Source: data/chunks.json
Each chunk carries: text, start, end, video_id, title, token_count

In [15]:
with open("data/chunks.json") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 160 chunks


## 11. Create Chroma collection

In [16]:
collection = chroma_client.get_or_create_collection(
    name="procurement_scm",
    metadata={"hnsw:space": "cosine"}
)

## 12. Embed and upsert chunks

Each chunk gets embedded and stored with its metadata (video_id, title,
timestamps) so retrieved results can be traced back to source + citation.

In [17]:
def embed_text(text: str) -> list[float]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


batch_size = 50

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    ids = [f"{c['video_id']}_{i+j}" for j, c in enumerate(batch)]
    texts = [c["text"] for c in batch]
    embeddings = [embed_text(t) for t in texts]
    metadatas = [
        {
            "video_id": c["video_id"],
            "title": c["title"],
            "start": c["start"],
            "end": c["end"]
        }
        for c in batch
    ]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas
    )

    print(f"Upserted {i + len(batch)}/{len(chunks)}")

print("Done embedding and indexing.")

Upserted 50/160
Upserted 100/160
Upserted 150/160
Upserted 160/160
Done embedding and indexing.


## 13. Sanity check — test a retrieval query

In [18]:
def test_query(query, n_results=3):
    results = collection.query(
        query_embeddings=[embed_text(query)],
        n_results=n_results
    )
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"--- {meta['title']} ({meta['start']:.0f}s) — dist={dist:.3f} ---")
        print(doc)
        print()

test_query("what are the stages of contract management?")

--- Contract Management in Procurement | Stages & Tools (59s) — dist=0.250 ---
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM platforms for tracking execution. Stage 3. Contract monitoring and com

In [19]:
test_query("how do you segment suppliers by importance?")

--- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (1870s) — dist=0.391 ---
From that, well, then you can be able to understand how do you manage that particular supplier in that particular item of your portfolio? What we believe is important is that you're not just focusing on segmenting your suppliers, but that you're automating that segmentation. Technology can help to guide that. But of course, the parameters need to be set by the business together with your organization, in your teams, and of course, aligned with your strategic goals. This automated segmentation, though, is a rule based setup that we do together with our customers. And you should be able to, of course, ascertain this with any technology that you would be finding out there on the market for supplier relationship management. It's a kind of if-then methodology, which is then allowing our customers to quickly segment their supplier bases, approve them, not approve them, but also then b

In [20]:
test_query("what causes stockouts and how is inventory tracked?")

--- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (176s) — dist=0.487 ---
Inventory management can be the heartbeat but of a supply chain or a company and its supply chain, especially because it plays a very important role in optimizing that all the different various aspects of a supply chain and its operations run as smoothly as possible and can actually function. So now we're going to take a look at some of these operations or what is within inventory management. Now look at those next. Now some of the aspects we're going to talk about is balancing supply and demand. I mentioned that a little bit earlier, but we're going to talk about it here too. Inventory management is going to be all about finding the perfect balance between supply and demand. Maintaining the right level of inventory, companies or organizations can ensure that the products are readily available when customers need them. This reduces stockouts, which can lead to lost sales opportuniti

## 14. Basic QA chain

Takes a user question → retrieves top-k relevant chunks from Chroma →
passes them as context to an LLM → returns a grounded answer with
source citations (video title + timestamp).

In [21]:
def answer_question(question: str, n_results: int = 4) -> dict:
    """
    Retrieve relevant chunks and generate an answer grounded in them.

    Returns a dict with the answer text and the source chunks used,
    so citations can be shown alongside the response.
    """
    results = collection.query(
        query_embeddings=[embed_text(question)],
        n_results=n_results
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context = "\n\n".join(
        f"[Source: {m['title']} at {m['start']:.0f}s]\n{d}"
        for d, m in zip(docs, metas)
    )

    system_prompt = (
        "You are a procurement and supply chain management assistant. "
        "Answer the user's question using ONLY the provided context. "
        "If the context doesn't contain enough information to answer, say so. "
        "Cite the video title when referencing specific information."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0.2
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [
            {"title": m["title"], "start": m["start"], "video_id": m["video_id"]}
            for m in metas
        ]
    }

## 15. Test the full pipeline

In [22]:
result = answer_question("What are the four stages of contract management?")

print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"- {s['title']} ({s['start']:.0f}s) — youtube.com/watch?v={s['video_id']}&t={int(s['start'])}")

ANSWER:
The four stages of contract management are:

1. Contract creation and negotiation: Identify procurement needs and draft contract terms including deliverables, pricing, and timelines, collaborating with legal and finance teams for term approvals.

2. Contract execution: Finalize and sign agreements through digital or physical signatures, route for internal approvals, and record contracts in a central repository.

3. Contract monitoring and compliance: Track SLAs, renewal dates, and non-compliance events using dashboards, and run audits to check deliverables against timelines.

4. Contract renewal or closure: Evaluate supplier performance and determine whether to extend or terminate the contract, archive closed contracts, and collect post-mortem lessons. 

This information is sourced from "Contract Management in Procurement | Stages & Tools."

SOURCES:
- Contract Management in Procurement | Stages & Tools (59s) — youtube.com/watch?v=_l-rmPwyv28&t=58
- Contract Management in Procu

## 16. Add conversational memory

Track the running conversation so follow-up questions can reference
earlier turns without the user re-stating context.

In [23]:
conversation_history = []

def answer_question_with_memory(question: str, n_results: int = 4) -> dict:
    results = collection.query(
        query_embeddings=[embed_text(question)],
        n_results=n_results
    )
    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context = "\n\n".join(
        f"[Source: {m['title']} at {m['start']:.0f}s]\n{d}"
        for d, m in zip(docs, metas)
    )

    system_prompt = (
        "You are a procurement and supply chain management assistant. "
        "Answer using ONLY the provided context. If the context is "
        "insufficient, say so. Cite the video title when referencing "
        "specific information. Use the conversation history to "
        "understand follow-up questions."
    )

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(conversation_history)
    messages.append({
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {question}"
    })

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2
    )

    answer = response.choices[0].message.content

    # store the turn (without the bulky retrieved context, just Q&A)
    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": answer})

    return {
        "answer": answer,
        "sources": [{"title": m["title"], "start": m["start"], "video_id": m["video_id"]} for m in metas]
    }

## 17. Test memory with a follow-up

In [24]:
r1 = answer_question_with_memory("What are the four stages of contract management?")
print(r1["answer"])

print("\n---\n")

r2 = answer_question_with_memory("Can you explain the second one in more detail?")
print(r2["answer"])

The four stages of contract management are:

1. **Contract creation and negotiation** - Identify procurement needs and draft contract terms including deliverables, pricing, and timelines.
2. **Contract execution** - Finalize and sign agreements through digital or physical signatures and record contracts in a central repository.
3. **Contract monitoring and compliance** - Track SLAs, renewal dates, and non-compliance events using dashboards and run audits to check deliverables against timelines.
4. **Contract renewal or closure** - Evaluate supplier performance and determine whether to extend or terminate the contract, archive closed contracts, and collect postmortem lessons. 

(Source: Contract Management in Procurement | Stages & Tools)

---

The second stage of contract management is **Contract execution**. This stage involves finalizing and signing agreements, which can be done through digital or physical signatures. It is crucial to ensure that all parties involved have agreed to t